In [1]:
pip install ultralytics 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\vyshnavpradeep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import shutil
import zipfile
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import random

In [6]:
os.makedirs("extracted/healthy", exist_ok=True)
os.makedirs("extracted/diseased", exist_ok=True)

healthy_zip = input("Enter path to healthy images zip: ")
diseased_zip = input("Enter path to diseased images zip: ")

with zipfile.ZipFile(healthy_zip, 'r') as z:
    z.extractall("extracted/healthy")

with zipfile.ZipFile(diseased_zip, 'r') as z:
    z.extractall("extracted/diseased")

In [7]:
valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.tif', '.gif', '.svg', '.ico', '.heic', '.heif', '.avif', '.raw', '.cr2', '.nef', '.arw', '.dng', '.orf', '.rw2', '.pef', '.sr2', '.raf', '.jfif', '.pgm', '.ppm', '.pbm', '.pnm', '.exr', '.hdr', '.oip', '.snip', '.snag', '.scr','.html')
healthy_images = []
for root, dirs, files in os.walk("extracted/healthy"):
    for f in files:
        if f.lower().endswith(valid_ext):
            healthy_images.append(os.path.join(root, f))

diseased_images = []
for root, dirs, files in os.walk("extracted/diseased"):
    for f in files:
        if f.lower().endswith(valid_ext):
            diseased_images.append(os.path.join(root, f))

healthy_images.sort()
diseased_images.sort()

print(f"Healthy images found: {len(healthy_images)}")
print(f"Diseased images found: {len(diseased_images)}")

Healthy images found: 100
Diseased images found: 100


In [8]:
print("HEALTHY IMAGES - Navigate through images and select for VALIDATION (30 needed)")
print("Unchecked images will go to TRAIN set \n")

from io import BytesIO

healthy_selections = [False] * len(healthy_images)
current_healthy = [0]

img = Image.open(healthy_images[0])
buf = BytesIO()
img.save(buf, format='PNG')
buf.seek(0)

image_display = widgets.Image(value=buf.read(), format='png', layout=widgets.Layout(width='700px', height='500px', object_fit='contain'))
label = widgets.HTML(value=f"<b style='font-size:16px;'>Image 0 / {len(healthy_images)-1}: {os.path.basename(healthy_images[0])}</b>")
cb = widgets.Checkbox(value=False, description='Select for VALIDATION', indent=False, layout=widgets.Layout(width='300px'))
counter_label = widgets.HTML(value="<b style='color:red;font-size:16px;'>Selected for validation: 0 / 30</b>")

prev_btn = widgets.Button(description='◀ Previous', layout=widgets.Layout(width='150px'))
next_btn = widgets.Button(description='Next ▶', layout=widgets.Layout(width='150px'))
goto_input = widgets.IntText(value=0, description='Go to:', layout=widgets.Layout(width='200px'))
goto_btn = widgets.Button(description='Go', layout=widgets.Layout(width='80px'))

def load_image(index):
    img = Image.open(healthy_images[index])
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    image_display.value = buf.read()
    label.value = f"<b style='font-size:16px;'>Image {index} / {len(healthy_images)-1}: {os.path.basename(healthy_images[index])}</b>"
    cb.value = healthy_selections[index]
    count = sum(healthy_selections)
    color = "green" if count == 30 else "red"
    counter_label.value = f"<b style='color:{color};font-size:16px;'>Selected for validation: {count} / 30</b>"

def on_cb_change(change):
    healthy_selections[current_healthy[0]] = change['new']
    count = sum(healthy_selections)
    color = "green" if count == 30 else "red"
    counter_label.value = f"<b style='color:{color};font-size:16px;'>Selected for validation: {count} / 30</b>"

def on_prev(b):
    if current_healthy[0] > 0:
        current_healthy[0] -= 1
        goto_input.value = current_healthy[0]
        load_image(current_healthy[0])

def on_next(b):
    if current_healthy[0] < len(healthy_images) - 1:
        current_healthy[0] += 1
        goto_input.value = current_healthy[0]
        load_image(current_healthy[0])

def on_goto(b):
    idx = max(0, min(goto_input.value, len(healthy_images) - 1))
    current_healthy[0] = idx
    load_image(idx)

cb.observe(on_cb_change, names='value')
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
goto_btn.on_click(on_goto)

nav_bar = widgets.HBox([prev_btn, next_btn, goto_input, goto_btn])
display(widgets.VBox([counter_label, label, image_display, cb, nav_bar]))

HEALTHY IMAGES - Navigate through images and select for VALIDATION (30 needed)
Unchecked images will go to TRAIN set 



In [9]:
print("DISEASED IMAGES - Navigate through images and select for VALIDATION (30 needed)")
print("Unchecked images will go to TRAIN set\n")

from io import BytesIO

diseased_selections = [False] * len(diseased_images)
current_diseased = [0]

img = Image.open(diseased_images[0])
buf = BytesIO()
img.save(buf, format='PNG')
buf.seek(0)

image_display_d = widgets.Image(value=buf.read(), format='png', layout=widgets.Layout(width='700px', height='500px', object_fit='contain'))
label_d = widgets.HTML(value=f"<b style='font-size:16px;'>Image 0 / {len(diseased_images)-1}: {os.path.basename(diseased_images[0])}</b>")
cb_d = widgets.Checkbox(value=False, description='Select for VALIDATION', indent=False, layout=widgets.Layout(width='300px'))
counter_label_d = widgets.HTML(value="<b style='color:red;font-size:16px;'>Selected for validation: 0 / 30</b>")

prev_btn_d = widgets.Button(description='◀ Previous', layout=widgets.Layout(width='150px'))
next_btn_d = widgets.Button(description='Next ▶', layout=widgets.Layout(width='150px'))
goto_input_d = widgets.IntText(value=0, description='Go to:', layout=widgets.Layout(width='200px'))
goto_btn_d = widgets.Button(description='Go', layout=widgets.Layout(width='80px'))

def load_image_d(index):
    img = Image.open(diseased_images[index])
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    image_display_d.value = buf.read()
    label_d.value = f"<b style='font-size:16px;'>Image {index} / {len(diseased_images)-1}: {os.path.basename(diseased_images[index])}</b>"
    cb_d.value = diseased_selections[index]
    count = sum(diseased_selections)
    color = "green" if count == 30 else "red"
    counter_label_d.value = f"<b style='color:{color};font-size:16px;'>Selected for validation: {count} / 30</b>"

def on_cb_change_d(change):
    diseased_selections[current_diseased[0]] = change['new']
    count = sum(diseased_selections)
    color = "green" if count == 30 else "red"
    counter_label_d.value = f"<b style='color:{color};font-size:16px;'>Selected for validation: {count} / 30</b>"

def on_prev_d(b):
    if current_diseased[0] > 0:
        current_diseased[0] -= 1
        goto_input_d.value = current_diseased[0]
        load_image_d(current_diseased[0])

def on_next_d(b):
    if current_diseased[0] < len(diseased_images) - 1:
        current_diseased[0] += 1
        goto_input_d.value = current_diseased[0]
        load_image_d(current_diseased[0])

def on_goto_d(b):
    idx = max(0, min(goto_input_d.value, len(diseased_images) - 1))
    current_diseased[0] = idx
    load_image_d(idx)

cb_d.observe(on_cb_change_d, names='value')
prev_btn_d.on_click(on_prev_d)
next_btn_d.on_click(on_next_d)
goto_btn_d.on_click(on_goto_d)

nav_bar_d = widgets.HBox([prev_btn_d, next_btn_d, goto_input_d, goto_btn_d])
display(widgets.VBox([counter_label_d, label_d, image_display_d, cb_d, nav_bar_d]))

DISEASED IMAGES - Navigate through images and select for VALIDATION (30 needed)
Unchecked images will go to TRAIN set



In [10]:
healthy_val_indices = [i for i, v in enumerate(healthy_selections) if v]
healthy_train_indices = [i for i, v in enumerate(healthy_selections) if not v]

diseased_val_indices = [i for i, v in enumerate(diseased_selections) if v]
diseased_train_indices = [i for i, v in enumerate(diseased_selections) if not v]

print("=" * 50)
print("DATASET SPLIT SUMMARY")
print("=" * 50)

print(f"\n{'Class':<15} {'Train':<10} {'Valid':<10} {'Total':<10}")
print("-" * 45)
print(f"{'Healthy':<15} {len(healthy_train_indices):<10} {len(healthy_val_indices):<10} {len(healthy_train_indices) + len(healthy_val_indices):<10}")
print(f"{'Diseased':<15} {len(diseased_train_indices):<10} {len(diseased_val_indices):<10} {len(diseased_train_indices) + len(diseased_val_indices):<10}")
print("-" * 45)
print(f"{'Total':<15} {len(healthy_train_indices) + len(diseased_train_indices):<10} {len(healthy_val_indices) + len(diseased_val_indices):<10} {len(healthy_train_indices) + len(healthy_val_indices) + len(diseased_train_indices) + len(diseased_val_indices):<10}")

print("\n" + "=" * 50)
print("VALIDATION CHECK")
print("=" * 50)

all_ok = True

if len(healthy_train_indices) != 52:
    print(f"WARNING: Healthy TRAIN has {len(healthy_train_indices)} images (expected 52)")
    all_ok = False
else:
    print(f"Healthy TRAIN: {len(healthy_train_indices)} images - OK")

if len(healthy_val_indices) != 23:
    print(f"WARNING: Healthy VALID has {len(healthy_val_indices)} images (expected 23)")
    all_ok = False
else:
    print(f"Healthy VALID: {len(healthy_val_indices)} images - OK")

if len(diseased_train_indices) != 52:
    print(f"WARNING: Diseased TRAIN has {len(diseased_train_indices)} images (expected 52)")
    all_ok = False
else:
    print(f"Diseased TRAIN: {len(diseased_train_indices)} images - OK")

if len(diseased_val_indices) != 23:
    print(f"WARNING: Diseased VALID has {len(diseased_val_indices)} images (expected 23)")
    all_ok = False
else:
    print(f"Diseased VALID: {len(diseased_val_indices)} images - OK")

print("\n" + "=" * 50)
if all_ok:
    print("ALL CHECKS PASSED - Ready to create dataset!")
else:
    print("SOME CHECKS FAILED - Please go back and adjust your selections.")
print("=" * 50)

print("\nHealthy VALIDATION image indices:", healthy_val_indices)
print("Diseased VALIDATION image indices:", diseased_val_indices)

DATASET SPLIT SUMMARY

Class           Train      Valid      Total     
---------------------------------------------
Healthy         70         30         100       
Diseased        70         30         100       
---------------------------------------------
Total           140        60         200       

VALIDATION CHECK

SOME CHECKS FAILED - Please go back and adjust your selections.

Healthy VALIDATION image indices: [0, 1, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15, 21, 29, 31, 34, 35, 37, 39, 45, 47, 48, 49, 50, 51, 52, 53, 55, 90, 91]
Diseased VALIDATION image indices: [0, 1, 3, 4, 6, 7, 9, 14, 16, 27, 28, 33, 35, 47, 50, 52, 53, 55, 59, 60, 62, 69, 70, 71, 75, 76, 77, 78, 84, 89]


In [11]:
base = "dataset"

if os.path.exists(base):
    shutil.rmtree(base)

for split in ["train", "val"]:
    for cls in ["healthy", "diseased"]:
        os.makedirs(os.path.join(base, split, cls), exist_ok=True)

for i in healthy_train_indices:
    shutil.copy2(healthy_images[i], os.path.join(base, "train", "healthy"))

for i in healthy_val_indices:
    shutil.copy2(healthy_images[i], os.path.join(base, "val", "healthy"))

for i in diseased_train_indices:
    shutil.copy2(diseased_images[i], os.path.join(base, "train", "diseased"))

for i in diseased_val_indices:
    shutil.copy2(diseased_images[i], os.path.join(base, "val", "diseased"))

print("Dataset structure created!")
print("=" * 50)
print(f"train/healthy:  {len(os.listdir(os.path.join(base, 'train', 'healthy')))}")
print(f"train/diseased: {len(os.listdir(os.path.join(base, 'train', 'diseased')))}")
print(f"val/healthy:    {len(os.listdir(os.path.join(base, 'val', 'healthy')))}")
print(f"val/diseased:   {len(os.listdir(os.path.join(base, 'val', 'diseased')))}")
print("=" * 50)

Dataset structure created!
train/healthy:  70
train/diseased: 70
val/healthy:    30
val/diseased:   30


In [12]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=os.path.abspath("dataset"),
    epochs=100,
    imgsz=224,
    batch=16,
    device="cpu",
    project="runs/classify",
    name="healthy_vs_diseased",
    patience=15,
    lr0=0.001,
    lrf=0.01,
    optimizer="AdamW",
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0001,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.2,
    erasing=0.3,
    crop_fraction=0.8,
    auto_augment="randaugment",
    dropout=0.3
)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'crop_fraction' is deprecated and will be removed in the future.
Ultralytics 8.4.6  Python-3.11.9 torch-2.9.1+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\dataset, degrees=15.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, epochs=100, erasing=0.3, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, m

In [13]:
metrics = model.val()
print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")
print("=" * 50)

Ultralytics 8.4.6  Python-3.11.9 torch-2.9.1+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\dataset\train... found 140 images in 2 classes  
val: C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\dataset\val... found 60 images in 2 classes  
test: None...
val: Fast image access  (ping: 0.10.0 ms, read: 815.9742.3 MB/s, size: 660.6 KB)
val: Scanning C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\dataset\val... 60 images, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 3.2it/s 1.2s0.5s
                   all        0.9          1
Speed: 0.0ms preprocess, 3.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\runs\classify\val
VALIDATION RESULTS
Top-1 Accuracy: 0.9000
Top-5 Accuracy: 1.0000


In [14]:
from IPython.display import Image as IPImage, display

results_dir = os.path.join("runs", "classify", "healthy_vs_diseased")

results_img = os.path.join(results_dir, "results.png")
if os.path.exists(results_img):
    print("TRAINING CURVES")
    display(IPImage(filename=results_img))

confusion_img = os.path.join(results_dir, "confusion_matrix.png")
if os.path.exists(confusion_img):
    print("CONFUSION MATRIX")
    display(IPImage(filename=confusion_img))

confusion_norm_img = os.path.join(results_dir, "confusion_matrix_normalized.png")
if os.path.exists(confusion_norm_img):
    print("NORMALIZED CONFUSION MATRIX")
    display(IPImage(filename=confusion_norm_img))

In [15]:
save_dir = r"C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\saved_models_(4)"
os.makedirs(save_dir, exist_ok=True)

best_model_path =r"C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\runs\classify\runs\classify\healthy_vs_diseased\weights\best.pt"
save_path = os.path.join(save_dir, "original.pt")

shutil.copy2(best_model_path, save_path)

print(f"Model saved to: {save_path}")

Model saved to: C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\saved_models_(4)\original.pt


In [16]:
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
from IPython.display import display
import ipywidgets as widgets
import os

loaded_model = YOLO(r"C:\Users\vyshnavpradeep\OneDrive\Desktop\miniproject\saved_models_(4)\model(4).pt")

test_folder = input("Enter path to test folder containing images: ")

valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.tif', '.gif', '.svg', '.ico', '.heic', '.heif', '.avif', '.raw', '.cr2', '.nef', '.arw', '.dng', '.orf', '.rw2', '.pef', '.sr2', '.raf', '.jfif', '.pgm', '.ppm', '.pbm', '.pnm', '.exr', '.hdr', '.oip', '.snip', '.snag', '.scr')

test_images = []
for f in sorted(os.listdir(test_folder)):
    if f.lower().endswith(valid_ext):
        test_images.append(os.path.join(test_folder, f))

print(f"Found {len(test_images)} images in test folder")

if len(test_images) > 20:
    print(f"Only first 20 images will be processed")
    test_images = test_images[:20]

image_widgets = []
healthy_count = 0
diseased_count = 0

for path in test_images:
    r = loaded_model.predict(source=path, imgsz=224, verbose=False)[0]
    pred = r.names[r.probs.top1]
    conf = r.probs.top1conf.item()

    if pred == "healthy":
        healthy_count += 1
    else:
        diseased_count += 1

    img = Image.open(path).convert("RGB")
    img = img.resize((400, 400))
    draw = ImageDraw.Draw(img)

    color = (0, 200, 0) if pred == "healthy" else (255, 0, 0)
    text = f"{pred.upper()} ({conf*100:.1f}%)"

    try:
        font = ImageFont.truetype("arial.ttf", 28)
    except:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    draw.rectangle([0, 0, text_w + 20, text_h + 20], fill=color)
    draw.text((10, 5), text, fill=(255, 255, 255), font=font)
    draw.rectangle([0, 0, 399, 399], outline=color, width=5)

    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)

    img_widget = widgets.Image(value=buf.read(), format='png', layout=widgets.Layout(width='400px', height='400px'))
    label_widget = widgets.HTML(value=f"<p style='text-align:center;font-size:12px;font-weight:bold;'>{os.path.basename(path)}</p>")
    image_widgets.append(widgets.VBox([img_widget, label_widget]))

rows = []
for i in range(0, len(image_widgets), 3):
    row = widgets.HBox(image_widgets[i:i+3], layout=widgets.Layout(justify_content='center'))
    rows.append(row)

print("=" * 60)
print(f"BATCH PREDICTION RESULTS - {len(test_images)} IMAGES")
print("=" * 60)

display(widgets.VBox(rows))

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Healthy:  {healthy_count}/{len(test_images)}")
print(f"Diseased: {diseased_count}/{len(test_images)}")
print("=" * 60)

Found 13 images in test folder
BATCH PREDICTION RESULTS - 13 IMAGES



SUMMARY
Healthy:  7/13
Diseased: 6/13
